<a href="https://colab.research.google.com/github/AnirudhPhophalia/1024160021_UCS420_CC_Assignments/blob/main/cc_ac4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Q1: Build Your Personalized Knowledge Base:

Take your college roll number.

Extract its digits. Build a pandas DataFrame with exactly 6 FAQ entries: 4 fixed entries (given in table below) and other 2 entries constructed from your own roll number digits as follows:  

• Take the LAST TWO DIGITS of your roll number. For each digit d, compute category = ["billing", "account", "general"][d % 3].

Invent one realistic question+answer+3 keywords per entry that fits the ssigned category (e.g. if d%3 gives "account", write a question like “how do I update my registered mobile number”).  

• # Example roll number ...23 -> digits 2, 3

• # digit 2 -> category[2 % 3] = general

• # digit 3 -> category[3 % 3] = billing

In [6]:
roll_no = 1024160021

In [7]:
import pandas as pd

fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.", "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings > Reset Password.", "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.", "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.", "keywords": "pay payment upi fee", "category": "billing"},
]

last_two_digits_str = str(roll_no)[-2:]
digit1 = int(last_two_digits_str[0])
digit2 = int(last_two_digits_str[1])

categories = ["billing", "account", "general"]

category1 = categories[digit1 % 3]
category2 = categories[digit2 % 3]

custom_entries = []

if category1 == "billing":
    custom_entries.append({
        "question": f"What is the refund policy for digit {digit1} related payments?",
        "answer": "Refunds for payments related to digit 2 are processed within 5-7 business days after approval.",
        "keywords": "refund policy payment",
        "category": category1
    })
elif category1 == "account":
    custom_entries.append({
        "question": f"How can I update my profile information for digit {digit1} account?",
        "answer": "You can update your profile by logging into your account and navigating to 'Profile Settings'.",
        "keywords": "profile update account",
        "category": category1
    })
elif category1 == "general":
    custom_entries.append({
        "question": f"Where can I find general information about digit {digit1} services?",
        "answer": "General information about our services can be found in the 'Help & Support' section of our website.",
        "keywords": "general info help",
        "category": category1
    })

if category2 == "billing":
    custom_entries.append({
        "question": f"Is there a discount for early payment for digit {digit2} invoices?",
        "answer": "Yes, a 10% discount is applied to invoices related to digit 0 if paid within the first 7 days.",
        "keywords": "discount early payment",
        "category": category2
    })
elif category2 == "account":
    custom_entries.append({
        "question": f"Can I link multiple accounts for digit {digit2} services?",
        "answer": "You can link up to 3 accounts for seamless management by visiting the 'Linked Accounts' section.",
        "keywords": "link accounts multiple",
        "category": category2
    })
elif category2 == "general":
    custom_entries.append({
        "question": f"What are the operating hours for digit {digit2} support?",
        "answer": "Our support team for services related to digit 0 is available Monday to Friday, 9 AM to 6 PM.",
        "keywords": "support hours contact",
        "category": category2
    })

all_entries = fixed_entries + custom_entries

df_faq = pd.DataFrame(all_entries)

display(df_faq)

,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
1,how to reset password,Go to Settings > Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,Where can I find general information about dig...,General information about our services can be ...,general info help,general
5,Is there a discount for early payment for digi...,"Yes, a 10% discount is applied to invoices rel...",discount early payment,billing


## Q2: Generate and Score a Hypothesis - Implement a scoring function that takes a query string and returns all matching entries ranked by confidence.

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

def score_query(query_string, df):
    # Combine 'question' and 'keywords' for better matching
    df['combined_text'] = df['question'] + ' ' + df['keywords']

    # Initialize TF-IDF Vectorizer
    tfidf_vectorizer = TfidfVectorizer(stop_words='english')

    # Fit and transform the combined text of FAQ entries
    tfidf_matrix = tfidf_vectorizer.fit_transform(df['combined_text'])

    # Transform the query string
    query_vector = tfidf_vectorizer.transform([query_string])

    # Compute cosine similarity between query and FAQ entries
    cosine_similarities = linear_kernel(query_vector, tfidf_matrix).flatten()

    # Create a Series of scores and associate with DataFrame index
    scores = pd.Series(cosine_similarities, index=df.index)

    # Rank entries by score in descending order
    ranked_entries = df.loc[scores.sort_values(ascending=False).index]
    ranked_entries['confidence_score'] = scores.sort_values(ascending=False)

    # Drop the temporary combined_text column
    df.drop(columns=['combined_text'], inplace=True)

    return ranked_entries


# Demonstrate the scoring function with a sample query
print("\n--- Demonstrating Q2 Scoring Function ---")
sample_query = "reset my password"
results = score_query(sample_query, df_faq.copy())
display(results[['question', 'category', 'confidence_score']].head())

sample_query_2 = "annual fee payment"
results_2 = score_query(sample_query_2, df_faq.copy())
display(results_2[['question', 'category', 'confidence_score']].head())


--- Demonstrating Q2 Scoring Function ---


,question,category,confidence_score
1,how to reset password,account,0.942809
0,what is the annual fee,billing,0.000000
2,what are your working hours,general,0.000000
3,how can i pay the fee,billing,0.000000
4,Where can I find general information about dig...,general,0.000000


,question,category,confidence_score
0,what is the annual fee,billing,0.592044
3,how can i pay the fee,billing,0.455563
5,Is there a discount for early payment for digi...,billing,0.249787
1,how to reset password,account,0.000000
2,what are your working hours,general,0.000000


## Q3: Write a function `same_category(category_name, df)` that returns all questions belonging to a given category. Call it using the category of one of the personalized entries from Q1, and print the result.

In [9]:
def same_category(category_name, df):
    """Returns all questions belonging to a given category."""
    return df[df['category'] == category_name][['question', 'category']]

# Call the function using the category of one of the personalized entries from Q1
# From Q1, digit1 resulted in category1 = 'general'
print(f"\n--- Demonstrating Q3 Function for category '{category1}' ---")
general_questions = same_category(category1, df_faq)
display(general_questions)

# From Q1, digit2 resulted in category2 = 'billing'
print(f"\n--- Demonstrating Q3 Function for category '{category2}' ---")
billing_questions = same_category(category2, df_faq)
display(billing_questions)


--- Demonstrating Q3 Function for category 'general' ---


,question,category
2,what are your working hours,general
4,Where can I find general information about dig...,general



--- Demonstrating Q3 Function for category 'billing' ---


,question,category
0,what is the annual fee,billing
3,how can i pay the fee,billing
5,Is there a discount for early payment for digi...,billing


## Q4: Pick any one entry in your knowledge base. Ask the user to input a new keyword, add it to that entry's keywords, and save your entire updated DataFrame to a CSV file named `<your_roll_number>_faq_data.csv`.

In [10]:
import ipywidgets as widgets
from IPython.display import display

# Choose an entry to update (e.g., the first entry)
entry_index = 0
selected_entry = df_faq.loc[entry_index]

print(f"Selected entry for update (Index {entry_index}):")
display(selected_entry[['question', 'keywords']])

def on_button_click(b):
    new_keyword = text_input.value.strip()
    if new_keyword:
        # Add the new keyword to the selected entry's keywords
        current_keywords = df_faq.loc[entry_index, 'keywords']
        updated_keywords = f"{current_keywords} {new_keyword}"
        df_faq.loc[entry_index, 'keywords'] = updated_keywords
        print(f"\nUpdated keywords for entry {entry_index}:")
        print(df_faq.loc[entry_index, 'keywords'])

        # Save the updated DataFrame to a CSV file
        output_filename = f"{roll_no}_faq_data.csv"
        df_faq.to_csv(output_filename, index=False)
        print(f"\nDataFrame saved to {output_filename}")
    else:
        print("\nNo keyword entered.")

text_input = widgets.Text(
    value='',
    placeholder='Type a new keyword here',
    description='New Keyword:',
    disabled=False
)

button = widgets.Button(description="Add Keyword & Save")
button.on_click(on_button_click)

display(text_input, button)

Selected entry for update (Index 0):


,0
question,what is the annual fee
keywords,fee cost price charge


Text(value='', description='New Keyword:', placeholder='Type a new keyword here')

Button(description='Add Keyword & Save', style=ButtonStyle())

## Q5: Using `groupby`, print how many FAQ entries you have per category.

In [11]:
print("\n--- Demonstrating Q5: FAQ Entries per Category ---")
category_counts = df_faq.groupby('category').size().reset_index(name='count')
display(category_counts)


--- Demonstrating Q5: FAQ Entries per Category ---


,category,count
0,account,1
1,billing,3
2,general,2


## Q6: Modify your Q2 scoring function so that if two or more entries tie for the highest score, it does not silently pick one — it prints all matching entries instead, so the user can see every equally good match. Demonstrate with one query that produces a tie (e.g. a query matching both "fee" entries) and one that doesn't.

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

def score_query_with_ties(query_string, df):
    # Combine 'question' and 'keywords' for better matching
    df['combined_text'] = df['question'] + ' ' + df['keywords']

    # Initialize TF-IDF Vectorizer
    tfidf_vectorizer = TfidfVectorizer(stop_words='english')

    # Fit and transform the combined text of FAQ entries
    tfidf_matrix = tfidf_vectorizer.fit_transform(df['combined_text'])

    # Transform the query string
    query_vector = tfidf_vectorizer.transform([query_string])

    # Compute cosine similarity between query and FAQ entries
    cosine_similarities = linear_kernel(query_vector, tfidf_matrix).flatten()

    # Create a Series of scores and associate with DataFrame index
    scores = pd.Series(cosine_similarities, index=df.index)

    # Get the maximum score
    max_score = scores.max()

    # Identify all entries that have the maximum score
    tied_entries_indices = scores[scores == max_score].index

    # Filter the DataFrame to include only the tied entries
    tied_ranked_entries = df.loc[tied_entries_indices].copy()
    tied_ranked_entries['confidence_score'] = scores.loc[tied_entries_indices]

    # Sort the original scores in descending order to get all ranked entries
    all_ranked_entries = df.loc[scores.sort_values(ascending=False).index].copy()
    all_ranked_entries['confidence_score'] = scores.sort_values(ascending=False)

    # Drop the temporary combined_text column
    df.drop(columns=['combined_text'], inplace=True)

    if len(tied_ranked_entries) > 1:
        print(f"\nMultiple entries tied for the highest score ({max_score:.4f}):")
        return tied_ranked_entries[['question', 'category', 'confidence_score']]
    else:
        return all_ranked_entries[['question', 'category', 'confidence_score']].head()


# Demonstrate with a query that produces a tie (e.g., matching both "fee" entries)
print("\n--- Demonstrating Q6 with a tie-producing query ---")
tie_query = "annual fee payment"
tie_results = score_query_with_ties(tie_query, df_faq.copy())
display(tie_results)

# Demonstrate with a query that doesn't produce a tie
print("\n--- Demonstrating Q6 with a non-tie-producing query ---")
no_tie_query = "reset password"
no_tie_results = score_query_with_ties(no_tie_query, df_faq.copy())
display(no_tie_results)


--- Demonstrating Q6 with a tie-producing query ---


,question,category,confidence_score
0,what is the annual fee,billing,0.592044
3,how can i pay the fee,billing,0.455563
5,Is there a discount for early payment for digi...,billing,0.249787
1,how to reset password,account,0.000000
2,what are your working hours,general,0.000000



--- Demonstrating Q6 with a non-tie-producing query ---


,question,category,confidence_score
1,how to reset password,account,0.942809
0,what is the annual fee,billing,0.000000
2,what are your working hours,general,0.000000
3,how can i pay the fee,billing,0.000000
4,Where can I find general information about dig...,general,0.000000
